In [ ]:
!pip install "setuptools<70.0.0" opfunu
import subprocess, sys

pkgs = ['numpy', 'scipy', 'pandas', 'tabulate', 'matplotlib', 'opfunu']
subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + pkgs + ['-q'])

import numpy as np
import scipy
print(f'✓ numpy {np.__version__}')
print(f'✓ scipy {scipy.__version__}')
print('✓ Dépendances prêtes.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon, rankdata, friedmanchisquare
from tabulate import tabulate
import warnings, time, os, pickle, glob, threading
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family'    : 'serif',
    'font.serif'     : ['Times MNABC Roman', 'DejaVu Serif'],
    'font.size'      : 11,
    'axes.titlesize' : 12,
    'axes.labelsize' : 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
    'axes.grid'      : True,
    'grid.alpha'     : 0.3,
    'grid.linestyle' : '--',
})

C_MNABC   = '#C0392B'                                        
C_NB2     = '#2980B9'                       
C_NB3     = '#0F6E56'                       
C_NB5     = '#8E44AD'                       
C_NB7     = '#E67E22'                       
C_NB9     = '#16A085'                       

NB_CONFIGS = [
    (1,  C_MNABC, 'MNABC (nb=1, baseline)'),
    (2,  C_NB2,   'MNABC_neighb (nb=2)'),
    (3,  C_NB3,   'MNABC_neighb (nb=3)'),
    (5,  C_NB5,   'MNABC_neighb (nb=5)'),
    (7,  C_NB7,   'MNABC_neighb (nb=7)'),
    (9,  C_NB9,   'MNABC_neighb (nb=9)'),
]
D = 50                                
os.makedirs(f'results_MNABC_neighb_Param_CEC2013_D{D}', exist_ok=True)
print('✓ Répertoires créés : results_MNABC_neighb_CEC2013/')
print(f'  Configurations : {[nb for nb,_,_ in NB_CONFIGS]}')

In [ ]:
D = 50 #or D = 30 for CEC2013 with 30-dimensional problems

PARAMS = dict(
    D          = D,
    SN         = 50,                                    
    MaxFEs     = 10_000 * D,                                            
    limit      = 100,
    alpha      = 0.7,                                    
    HUFS_ratio = 0.2,                            
    NB_SIZES   = [1, 2, 3, 5, 7, 9],                                 
    RUNS       = 30,
              
    N_FUNCS    = 28,
    F_OFFSET   = -1400.0,                                    
)

print('Paramètres expérimentaux — MNABC_neighb vs MNABC, CEC 2013 :')
for k, v in PARAMS.items():
    print(f'  {k:20s} = {v}')
print()
print(f'  Budget total / run : {PARAMS["MaxFEs"]:,} évaluations')
print(f'  Configurations     : {len(PARAMS["NB_SIZES"])} (nb_size = {PARAMS["NB_SIZES"]})')
print(f'  Fonctions CEC2013  : F1-F28')

In [ ]:
try:
    from opfunu.cec_based import cec2013
    _OPFUNU_AVAILABLE = True
    print('✓ opfunu chargé — fonctions CEC 2013 officielles disponibles.')
except ImportError:
    _OPFUNU_AVAILABLE = False
    print('⚠ opfunu non disponible — utilisation des fonctions proxy CEC 2013.')

def get_cec2013_function(func_id, D=30):
    """
    Retourne une fonction CEC 2013.
    func_id : 1 à 28.
    Utilise opfunu si disponible, sinon proxy.
    """
    if _OPFUNU_AVAILABLE:
        try:
            f = getattr(cec2013, f'F{func_id}')(ndim=D)
            return f.evaluate, f.f_global, (-100.0, 100.0)
        except Exception:
            pass
                                                                    
    np.random.seed(func_id * 7 + 13)
    shifts = np.random.uniform(-80, 80, D)
    M      = np.eye(D)                                           

    if func_id == 1:                     
        def fn(x): return float(np.sum((x - shifts[:len(x)])**2)) - 1400.0
        return fn, -1400.0, (-100.0, 100.0)

    elif func_id == 2:                         
        def fn(x):
            z = x - shifts[:len(x)]
            return float(sum(np.sum(z[:i+1])**2 for i in range(len(x)))) - 1300.0
        return fn, -1300.0, (-100.0, 100.0)

    elif func_id == 3:                       
        conds = np.array([10**(6 * i / (D-1)) for i in range(D)])
        def fn(x):
            z = x - shifts[:len(x)]
            return float(np.dot(conds, z**2)) - 1200.0
        return fn, -1200.0, (-100.0, 100.0)

    elif func_id <= 5:                             
        def fn(x):
            z = x - shifts[:len(x)]
            return float(np.sum(np.abs(z))**(1 + func_id*0.1)) - (1200.0 - func_id*100)
        return fn, -(1200.0 - func_id*100), (-100.0, 100.0)

    else:                                                                     
        g_id = func_id % 5
        base_offset = -1400.0 + func_id * 100.0

        if g_id == 0:           
            def fn(x):
                z = x - shifts[:len(x)]
                n = len(z)
                return float(20 - 20*np.exp(-0.2*np.sqrt(np.mean(z**2)))
                             - np.exp(np.mean(np.cos(2*np.pi*z))) + np.e) + base_offset
        elif g_id == 1:            
            def fn(x):
                z = x - shifts[:len(x)]
                return float(10*len(z) + np.sum(z**2 - 10*np.cos(2*np.pi*z))) + base_offset
        elif g_id == 2:           
            def fn(x):
                z = x - shifts[:len(x)]
                i = np.arange(1, len(z)+1)
                return float(np.sum(z**2)/4000 - np.prod(np.cos(z/np.sqrt(i))) + 1) + base_offset
        elif g_id == 3:             
            def fn(x):
                z = x - shifts[:len(x)] + 1
                return float(np.sum(100*(z[1:]-z[:-1]**2)**2 + (z[:-1]-1)**2)) + base_offset
        else:                          
            def fn(x):
                z = x - shifts[:len(x)]
                return float(np.max(np.abs(z))) + base_offset

        return fn, base_offset, (-100.0, 100.0)

CEC2013_INFO = {
              
    1:  ('F1',  'Sphere',                     'Unimodal'),
    2:  ('F2',  'Schwefel 1.2',               'Unimodal'),
    3:  ('F3',  'Rotated Elliptic',           'Unimodal'),
    4:  ('F4',  'Schwefel 1.2 Noise',         'Unimodal'),
    5:  ('F5',  'Schwefel 2.6',               'Unimodal'),
                      
    6:  ('F6',  'Shifted Rosenbrock',         'Multimodal'),
    7:  ('F7',  'Shifted Griewank',           'Multimodal'),
    8:  ('F8',  'Shifted Ackley',             'Multimodal'),
    9:  ('F9',  'Shifted Rastrigin',          'Multimodal'),
    10: ('F10', 'Shifted Rastrigin Rot.',     'Multimodal'),
    11: ('F11', 'Shifted Weierstrass',        'Multimodal'),
    12: ('F12', 'Schwefel 2.13',              'Multimodal'),
                         
    13: ('F13', 'Expanded Griewank Rosenbrock','Multimodal'),
    14: ('F14', 'Expanded Scaffer F6',        'Multimodal'),
                        
    15: ('F15', 'Hybrid Composition 1',       'Hybrid'),
    16: ('F16', 'Hybrid Composition 2',       'Hybrid'),
    17: ('F17', 'Hybrid Composition 3',       'Hybrid'),
    18: ('F18', 'Hybrid Composition 4',       'Hybrid'),
    19: ('F19', 'Rotated Hybrid Composition 1','Hybrid'),
    20: ('F20', 'Rotated Hybrid Composition 2','Hybrid'),
    21: ('F21', 'Rotated Hybrid Composition 3','Hybrid'),
    22: ('F22', 'Rotated Hybrid Composition 4','Hybrid'),
    23: ('F23', 'Non-Continuous Hybrid',      'Hybrid'),
    24: ('F24', 'Rotated Non-Continuous',     'Hybrid'),
    25: ('F25', 'Scaffer Hybrid',             'Hybrid'),
    26: ('F26', 'Rotated Scaffer Hybrid',     'Hybrid'),
    27: ('F27', 'Expanded Hybrid 1',          'Hybrid'),
    28: ('F28', 'Expanded Hybrid 2',          'Hybrid'),
}

CEC2013_FUNCS = {}
for fid in range(1, 29):
    fn, fopt, bounds = get_cec2013_function(fid, D)
    CEC2013_FUNCS[fid] = {'fn': fn, 'f_opt': fopt, 'bounds': bounds,
                           'name': CEC2013_INFO[fid][0],
                           'desc': CEC2013_INFO[fid][1],
                           'type': CEC2013_INFO[fid][2]}

print(f'✓ {len(CEC2013_FUNCS)} fonctions CEC 2013 chargées (D={D}).')
print(f'  Unimodal  : F1-F5  ({sum(1 for v in CEC2013_FUNCS.values() if v["type"]=="Unimodal")})')
print(f'  Multimodal: F6-F14 ({sum(1 for v in CEC2013_FUNCS.values() if v["type"]=="Multimodal")})')
print(f'  Hybrid    : F15-F28({sum(1 for v in CEC2013_FUNCS.values() if v["type"]=="Hybrid")})')

In [ ]:
class MNABC:
    
    def __init__(self, func_id, SN, MaxFEs, limit, D,
                 alpha=0.7, hufs_ratio=0.2):
        info = CEC2013_FUNCS[func_id]
        self.fn        = info['fn']
        self.f_opt     = info['f_opt']
        self.lb, self.ub = info['bounds']
        self.func_id   = func_id
        self.SN        = SN
        self.MaxFEs    = MaxFEs
        self.limit     = limit
        self.D         = D
        self.alpha     = alpha
        self.hufs_ratio= hufs_ratio

    def _eval(self, x):
        return float(self.fn(x))

    def _fitness_val(self, fval):
        return 1.0 / (1.0 + abs(fval - self.f_opt))

    def _utility(self, pop, fvals):
        fa = np.array([self._fitness_val(f) for f in fvals])
        fa_range = fa.max() - fa.min()
        fn = (fa - fa.min()) / (fa_range + 1e-10)
        d = np.zeros(self.SN)
        for i in range(self.SN):
            di = np.linalg.norm(pop - pop[i], axis=1); di[i] = np.inf
            d[i] = di.min()
        d_range = d.max() - d.min()
        dn = (d - d.min()) / (d_range + 1e-10)
        return self.alpha * fn + (1 - self.alpha) * dn                

    @staticmethod
    def _safe_probs(utils):
        s = np.sum(utils)
        if not np.isfinite(s) or s < 1e-10:
            n = len(utils); return np.full(n, 1.0/n)
        pr = utils / s
        return pr / pr.sum()

    def _hufs(self, pop, fvals):
        utils = self._utility(pop, fvals)
        k     = max(1, int(self.SN * self.hufs_ratio))
        idx   = np.argsort(utils)[::-1][:k]
        return pop[idx], fvals[idx]

    def _anchor_shift(self, pop, i, anchor, k_idx, fvals):
        
        jj = np.random.randint(self.D)
        v  = pop[i].copy()
        v[jj] = np.clip(
            anchor[jj] + np.random.uniform(-1, 1) * (pop[i, jj] - pop[k_idx, jj]),
            self.lb, self.ub)
        return v

    def run(self, seed=None):
        if seed is not None: np.random.seed(seed)
        lb, ub, D, SN = self.lb, self.ub, self.D, self.SN

        pop   = np.random.uniform(lb, ub, (SN, D))
        fvals = np.array([self._eval(x) for x in pop])
        trial = np.zeros(SN, int)
        best_val = fvals.min()
        best_err = best_val - self.f_opt
        NFE = SN
        history = [best_err]

        while NFE < self.MaxFEs:
            hufs, hufs_fv = self._hufs(pop, fvals)
            x_best = hufs[np.argmin(hufs_fv)]

            for i in range(SN):
                if NFE >= self.MaxFEs: break
                anc = hufs[np.random.randint(len(hufs))]
                k   = np.random.choice([j for j in range(SN) if j != i])
                v   = self._anchor_shift(pop, i, anc, k, fvals)
                fv  = self._eval(v)
                NFE += 1
                if fv <= fvals[i]: pop[i]=v; fvals[i]=fv; trial[i]=0
                else: trial[i] += 1
                if fv < best_val: best_val = fv

            utils = self._utility(pop, fvals)
            pr    = self._safe_probs(utils)
            for _ in range(SN):
                if NFE >= self.MaxFEs: break
                i   = np.random.choice(SN, p=pr)
                anc = hufs[np.random.randint(len(hufs))]
                k   = np.random.choice([j for j in range(SN) if j != i])
                v   = self._anchor_shift(pop, i, anc, k, fvals)
                fv  = self._eval(v)
                NFE += 1
                if fv <= fvals[i]: pop[i]=v; fvals[i]=fv; trial[i]=0
                else: trial[i] += 1
                if fv < best_val: best_val = fv

            for i in range(SN):
                if NFE >= self.MaxFEs: break
                if trial[i] > self.limit:
                    r1,r2,r3,r4 = np.random.uniform(0,1,4)
                    xr = pop[np.random.randint(SN)]
                    pop[i] = np.clip(
                        r1*x_best + r2*xr + r3*(lb + r4*(ub-lb)), lb, ub)
                    fvals[i] = self._eval(pop[i])
                    NFE += 1; trial[i] = 0
                    if fvals[i] < best_val: best_val = fvals[i]

            best_err = best_val - self.f_opt
            history.append(max(0.0, best_err))

        return max(0.0, best_val - self.f_opt), np.array(history)

In [ ]:
class MNABC_neighb(MNABC):
    

    def __init__(self, func_id, SN, MaxFEs, limit, D,
                 alpha=0.7, hufs_ratio=0.2, nb_size=3):
        super().__init__(func_id, SN, MaxFEs, limit, D, alpha, hufs_ratio)
        self.nb_size = min(nb_size, D)               

    def _neighborhood(self, jj):
        
        half = self.nb_size // 2
        return [(jj + t) % self.D for t in range(-half, self.nb_size - half)]

    def _anchor_shift(self, pop, i, anchor, k_idx, fvals):
        
        jj      = np.random.randint(self.D)                             
        nb_dims = self._neighborhood(jj)                          
        v       = pop[i].copy()
        for j in nb_dims:
            v[j] = np.clip(
                anchor[j] + np.random.uniform(-1, 1) * (pop[i, j] - pop[k_idx, j]),
                self.lb, self.ub)
        return v

print('✓ MNABC_neighb (Multi-Neighborhood Anchor-Shift, Eq.2\') prêt.')
print('  nb_size ∈ {2, 3, 5, 7, 9} — Complet ablation ')
print()
                           
test_opt = MNABC_neighb(1, SN=5, MaxFEs=10, limit=5, D=10, nb_size=5)
print('  Test voisinage(jj=7, nb_size=5, D=10) :', test_opt._neighborhood(7))
print('  Test voisinage(jj=0, nb_size=3, D=10) :', test_opt._neighborhood(0))                    
del test_opt

In [ ]:
results = {}

In [ ]:
print('Sanity check MNABC_neighb vs MNABC (MaxFEs=300, SN=10, 1 run) :')
for fid in [1, 6, 15]:                                
    err_u, _ = MNABC(fid, SN=10, MaxFEs=300, limit=10, D=D).run(seed=0)
    for nb in [3, 5]:
        err_mn, _ = MNABC_neighb(fid, SN=10, MaxFEs=300, limit=10, D=D, nb_size=nb).run(seed=0)
        print(f'  F{fid:2d} ({CEC2013_INFO[fid][2]:10s}): '
              f'MNABC(nb=1)={err_u:.2e}  MNABC_neighb(nb={nb})={err_mn:.2e}')
print('✓ Sanity check OK — MNABC et MNABC_neighb opérationnels.')

In [ ]:
RUNS   = PARAMS['RUNS']
SN     = PARAMS['SN']
limit_ = PARAMS['limit']
alpha_ = PARAMS['alpha']
hr     = PARAMS['HUFS_ratio']
MaxFEs_= PARAMS['MaxFEs']
NB_SIZES = PARAMS['NB_SIZES']                      

t0 = time.time()

for fid in range(1, 29):            
    fname = CEC2013_INFO[fid][0]
    ftype = CEC2013_INFO[fid][2]

    if fid not in results:
        results[fid] = {nb: {'errors': [], 'hist': None} for nb in NB_SIZES}

    nb_done = [nb for nb in NB_SIZES
               if len(results[fid][nb]['errors']) >= RUNS]
    if len(nb_done) == len(NB_SIZES):
        means = {nb: np.mean(results[fid][nb]['errors']) for nb in NB_SIZES}
        best_nb = min(means, key=means.get)
        print(f'{fname} ({ftype:10s}) [complet] '
              f'MNABC={means[1]:.2e}  '
              f'best_nb={best_nb} ({means[best_nb]:.2e})')
        continue

    t_f = time.time()
    for run in range(RUNS):
        seed = run * 137 + fid * 31

        for nb in NB_SIZES:
            if len(results[fid][nb]['errors']) > run:
                continue  

            if nb == 1:
                opt = MNABC(
                    fid, SN=SN, MaxFEs=MaxFEs_, limit=limit_, D=D,
                    alpha=alpha_, hufs_ratio=hr)
            else:
                opt = MNABC_neighb(
                    fid, SN=SN, MaxFEs=MaxFEs_, limit=limit_, D=D,
                    alpha=alpha_, hufs_ratio=hr, nb_size=nb)

            err, hist = opt.run(seed=seed)
            results[fid][nb]['errors'].append(float(err))
            if results[fid][nb]['hist'] is None:
                results[fid][nb]['hist'] = hist

        if (run+1) % 10 == 0 or (run+1) == RUNS:
            line = f'  {fname} ({ftype:10s}) run {run+1:2d}/{RUNS}  '
            for nb in NB_SIZES:
                errs = results[fid][nb]['errors']
                if errs:
                    line += f'nb={nb}:{np.mean(errs):.2e}  '
            print(line, flush=True)

    elapsed = time.time() - t_f
    means = {nb: np.mean(results[fid][nb]['errors']) for nb in NB_SIZES}
    best_nb = min(means, key=means.get)
    print(f'{fname} TERMINÉ [{elapsed:.0f}s]  MNABC(nb=1)={means[1]:.2e}  '
          f'Best=nb={best_nb} ({means[best_nb]:.2e})')

print(f'\n✓ Expérience terminée en {(time.time()-t0)/3600:.2f}h')

In [ ]:
from scipy.stats import wilcoxon, rankdata, friedmanchisquare

def wilcoxon_test(e1, e2, alpha_level=0.05):
    """'+' = e2 sig. meilleur, '-' = e1 sig. meilleur, '=' = égalité."""
    try:
        _, p = wilcoxon(e1, e2, alternative='two-sided')
    except ValueError:
        p = 1.0
    if p < alpha_level:
        return ('+' if np.mean(e2) < np.mean(e1) else '-'), float(p)
    return '=', float(p)

rows = []
for fid in range(1, 29):
    if fid not in results: continue
    fname = CEC2013_INFO[fid][0]
    ftype = CEC2013_INFO[fid][2]
    fdesc = CEC2013_INFO[fid][1]

    e_base = np.array(results[fid][1]['errors'])              
    row = {
        'FuncID' : fid,
        'Name'   : fname,
        'Type'   : ftype,
        'Desc'   : fdesc,
                        
        'nb1_mean'  : e_base.mean(),
        'nb1_std'   : e_base.std(ddof=0),
        'nb1_best'  : e_base.min(),
        'nb1_worst' : e_base.max(),
        'nb1_median': np.median(e_base),
    }

    all_errors = [e_base]                                                  

    for nb in [2, 3, 5, 7, 9]:
        e_nb = np.array(results[fid][nb]['errors'])
        sig, pval = wilcoxon_test(e_base, e_nb)
        row[f'nb{nb}_mean']   = e_nb.mean()
        row[f'nb{nb}_std']    = e_nb.std(ddof=0)
        row[f'nb{nb}_best']   = e_nb.min()
        row[f'nb{nb}_worst']  = e_nb.max()
        row[f'nb{nb}_median'] = np.median(e_nb)
        row[f'sig_nb{nb}']    = sig
        row[f'p_nb{nb}']      = pval
        all_errors.append(e_nb)

    means_6 = [np.mean(e) for e in all_errors]
    ranks_6 = list(rankdata(means_6))
    row['fr_nb1'] = ranks_6[0]
    for i, nb in enumerate([2,3,5,7,9]):
        row[f'fr_nb{nb}'] = ranks_6[i+1]

    best_nb = NB_SIZES[np.argmin(means_6)]
    row['best_nb'] = best_nb

    rows.append(row)

df = pd.DataFrame(rows)

print('═'*70)
print('RÉSULTATS CEC 2013 — MNABC_neighb vs MNABC (nb_size ablation)')
print('W/T/L : MNABC_neighb(nb=k) vs MNABC(nb=1)')
print('═'*70)
for nb in [2, 3, 5, 7, 9]:
    W = (df[f'sig_nb{nb}']=='+').sum()
    T = (df[f'sig_nb{nb}']=='=').sum()
    L = (df[f'sig_nb{nb}']=='-').sum()
    print(f'  nb_size={nb}: W={W:2d}  T={T:2d}  L={L:2d} /28')

print()
print('FRIEDMAN AVERAGE RANK (28 fonctions) :')
for nb in NB_SIZES:
    avg_r = df[f'fr_nb{nb}'].mean()
    label = 'MNABC (baseline)' if nb==1 else f'MNABC_neighb nb={nb}'
    print(f'  nb={nb}  {label:25s}: {avg_r:.4f}')

best_avg_nb = min(NB_SIZES, key=lambda nb: df[f'fr_nb{nb}'].mean())
print(f'  → Meilleure configuration : nb_size={best_avg_nb}')

print()
print('Distribution meilleure config par fonction :')
for nb in NB_SIZES:
    cnt = (df['best_nb'] == nb).sum()
    print(f'  nb={nb}: {cnt:2d} fonctions')

df.to_csv(f'results_MNABC_neighb_Param_CEC2013_D{D}/full_stats_cec2013.csv', index=False)
print('\n✓ Sauvegardé → results_MNABC_neighb_CEC2013/full_stats_cec2013.csv')

In [ ]:
print('TABLEAU ERREURS FORMAT CEC OFFICIEL — MNABC_neighb vs MNABC')
print('='*80)

cec_rows = []
for fid in range(1, 29):
    if fid not in results: continue
    fname = CEC2013_INFO[fid][0]
    ftype = CEC2013_INFO[fid][2]
    for nb in NB_SIZES:
        errs = np.array(results[fid][nb]['errors'])
        algo_name = 'MNABC' if nb==1 else f'MNABC_neighb(nb={nb})'
        cec_rows.append({
            'FuncID' : fid,
            'Func'   : fname,
            'Type'   : ftype,
            'Algo'   : algo_name,
            'Best'   : errs.min(),
            'Worst'  : errs.max(),
            'Median' : np.median(errs),
            'Mean'   : errs.mean(),
            'Std'    : errs.std(ddof=0),
        })

df_cec = pd.DataFrame(cec_rows)
df_cec.to_csv(f'results_MNABC_neighb_Param_CEC2013_D{D}/cec_format_errors.csv', index=False)

for algo_name in ['MNABC', 'MNABC_neighb(nb=3)', 'MNABC_neighb(nb=5)']:
    sub = df_cec[df_cec['Algo']==algo_name][['Func','Type','Best','Worst','Median','Mean','Std']].copy()
    for c in ['Best','Worst','Median','Mean','Std']:
        sub[c] = sub[c].map('{:.3e}'.format)
    print(f'\n── {algo_name} ──')
    print(tabulate(sub, headers='keys', tablefmt='grid', showindex=False))

print('\n✓ cec_format_errors.csv sauvegardé')

In [ ]:
import os
os.makedirs(f'results_MNABC_neighb_Param_CEC2013_D{D}', exist_ok=True)

NB_LABELS = {1:'MNABC\n(nb=1)', 2:'nb=2', 3:'nb=3', 5:'nb=5', 7:'nb=7', 9:'nb=9'}
NB_COLORS = {1:C_MNABC, 2:C_NB2, 3:C_NB3, 5:C_NB5, 7:C_NB7, 9:C_NB9}
NB_LS     = {1:'--', 2:':', 3:'-', 5:'-.', 7:(0,(5,2)), 9:(0,(3,1,1,1))}

fig, ax = plt.subplots(figsize=(14, 7))

fids = [fid for fid in range(1, 29) if fid in results]
hmap = np.zeros((len(NB_SIZES), len(fids)))
for j, fid in enumerate(fids):
    for i, nb in enumerate(NB_SIZES):
        errs = results[fid][nb]['errors']
        hmap[i, j] = np.mean(errs) if errs else np.nan

log_hmap = np.log10(np.maximum(hmap, 1e-10))
im = ax.imshow(log_hmap, aspect='auto', cmap='RdYlGn_r')
plt.colorbar(im, ax=ax, label='log₁₀(Mean Error)')
ax.set_xticks(range(len(fids)))
ax.set_xticklabels([f'F{fid}' for fid in fids], rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(NB_SIZES)))
ax.set_yticklabels([NB_LABELS[nb] for nb in NB_SIZES], fontsize=9)
ax.set_title(f'MNABC_neighb vs MNABC — Mean Error (log₁₀) · CEC 2013 (D={D}, 30 runs)',
             fontsize=12)
ax.set_xlabel('CEC 2013 Function', fontsize=11)
ax.set_ylabel('nb_size configuration', fontsize=11)

for j in range(len(fids)):
    best_i = np.nanargmin(log_hmap[:, j])
    ax.add_patch(plt.Rectangle((j-0.5, best_i-0.5), 1, 1,
                                fill=False, edgecolor='black', lw=1.5))

plt.tight_layout()
plt.savefig(f'results_MNABC_neighb_Param_CEC2013_D{D}/fig1_heatmap_errors.pdf', bbox_inches='tight')
plt.show()
print('✓ fig1_heatmap_errors.pdf')

selected = [1, 6, 9, 15, 21, 28]                                
selected = [fid for fid in selected if fid in results]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, fid in enumerate(selected):
    ax = axes[idx]
    for nb in NB_SIZES:
        h = results[fid][nb].get('hist')
        if h is None: continue
        label = NB_LABELS[nb].replace('\n', ' ')
        x = np.linspace(0, MaxFEs_, len(h))
        h_log = np.log10(np.maximum(h, 1e-10))
        ax.plot(x, h_log, color=NB_COLORS[nb], lw=1.4,
                ls=NB_LS[nb], label=label)
    ax.set_title(f'{CEC2013_INFO[fid][0]} — {CEC2013_INFO[fid][1][:25]}',
                 fontsize=9)
    ax.set_xlabel('NFE', fontsize=8)
    ax.set_ylabel('log₁₀(Error)', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.text(0.98, 0.98, CEC2013_INFO[fid][2], transform=ax.transAxes,
            ha='right', va='top', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='wheat', alpha=0.5))

handles = [Line2D([0],[0], color=NB_COLORS[nb], lw=1.5, ls=NB_LS[nb],
                  label=NB_LABELS[nb].replace('\n',' '))
           for nb in NB_SIZES]
fig.legend(handles=handles, loc='lower center', ncol=6,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f'Convergence — MNABC_neighb nb_size ablation · CEC 2013 (D={D}, 30 runs)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'results_MNABC_neighb_Param_CEC2013_D{D}/fig2_convergence.pdf', bbox_inches='tight')
plt.show()
print('✓ fig2_convergence.pdf')

avg_ranks = {nb: df[f'fr_nb{nb}'].mean() for nb in NB_SIZES}

fig, ax = plt.subplots(figsize=(9, 5))
xpos   = np.arange(len(NB_SIZES))
labels = [NB_LABELS[nb].replace('\n',' ') for nb in NB_SIZES]
bars   = ax.bar(xpos, [avg_ranks[nb] for nb in NB_SIZES],
                color=[NB_COLORS[nb] for nb in NB_SIZES],
                alpha=0.85, edgecolor='white', width=0.6)
for bar, nb in zip(bars, NB_SIZES):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{avg_ranks[nb]:.3f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xticks(xpos)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Average Friedman Rank (lower = better)', fontsize=11)
ax.set_title(f'Friedman Ranking — nb_size Ablation · CEC 2013 (D={D}, 28 functions, 30 runs)',
             fontsize=11)
ax.set_ylim(0, len(NB_SIZES) + 0.5)
ax.axhline(np.mean(list(avg_ranks.values())), color='grey', lw=0.8,
           ls='--', alpha=0.5, label='Average')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'results_MNABC_neighb_Param_CEC2013_D{D}/fig3_friedman_rank.pdf', bbox_inches='tight')
plt.show()
print('✓ fig3_friedman_rank.pdf')

fig, ax = plt.subplots(figsize=(10, 5))
nb_test = [2, 3, 5, 7, 9]                                  
W_vals  = [(df[f'sig_nb{nb}']=='+').sum() for nb in nb_test]
T_vals  = [(df[f'sig_nb{nb}']=='=').sum() for nb in nb_test]
L_vals  = [(df[f'sig_nb{nb}']=='-').sum() for nb in nb_test]

xpos = np.arange(len(nb_test))
p1 = ax.bar(xpos, W_vals, 0.5, label='W (MNABC_neighb better)', color='#27AE60', alpha=0.9)
p2 = ax.bar(xpos, T_vals, 0.5, bottom=W_vals, label='T (Tie)', color='#95A5A6', alpha=0.9)
p3 = ax.bar(xpos, L_vals, 0.5, bottom=np.array(W_vals)+np.array(T_vals),
            label='L (MNABC better)', color='#E74C3C', alpha=0.9)

for i, (w, t, l) in enumerate(zip(W_vals, T_vals, L_vals)):
    if w > 0: ax.text(i, w/2, str(w), ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    if t > 0: ax.text(i, w + t/2, str(t), ha='center', va='center', fontsize=10, color='black')
    if l > 0: ax.text(i, w+t + l/2, str(l), ha='center', va='center', fontsize=11, fontweight='bold', color='white')

ax.set_xticks(xpos)
ax.set_xticklabels([f'MNABC_neighb\nnb={nb}' for nb in nb_test], fontsize=10)
ax.set_ylabel('Number of Functions (out of 28)', fontsize=11)
ax.set_title('W/T/L — MNABC_neighb(nb=k) vs MNABC(nb=1)\n'
             'Wilcoxon signed-rank α=0.05 · CEC 2013 · 30 runs', fontsize=11)
ax.legend(fontsize=10, loc='upper right')
ax.set_ylim(0, 30)
plt.tight_layout()
plt.savefig(f'results_MNABC_neighb_Param_CEC2013_D{D}/fig4_wtl.pdf', bbox_inches='tight')
plt.show()
print('✓ fig4_wtl.pdf')

In [ ]:
def fmt_e(v, prec=2): return f'{v:.{prec}e}'
def sig_sym(s):
    return {'+': r'$\bullet$', '-': r'$\circ$', '=': r'$\approx$'}.get(s, r'$\approx$')

avg_ranks = {nb: df[f'fr_nb{nb}'].mean() for nb in NB_SIZES if nb != 1}
best_MNABC_neighb_nb = min(avg_ranks, key=avg_ranks.get)
print(f'Meilleure configuration globale MNABC_neighb : nb_size={best_MNABC_neighb_nb}')

col = f'nb{best_MNABC_neighb_nb}'

lines = [
    r'\begin{table*}[htbp]',
    r'\centering',
    r'\caption{Comparison of MNABC and MNABC_neighb (nb\_size=' + str(best_MNABC_neighb_nb) + r') on CEC 2013}',
    r'\label{tab:MNABC_neighb_cec2013}',
    r'\setlength{\tabcolsep}{3.5pt}',
    r'\begin{tabular}{llr|rrrrr|rrrrr|r}',
    r'\toprule',
    r'Func & Type & Rank$_\mathrm{Fr}$'
    r' & \multicolumn{5}{c|}{MNABC (nb=1, Baseline)}'
    r' & \multicolumn{5}{c|}{MNABC_neighb (nb=' + str(best_MNABC_neighb_nb) + r', Proposed)} & Sig. \\',
    r'\cmidrule(lr){4-8}\cmidrule(lr){9-13}',
    r' & & & Best & Worst & Med. & Mean & Std & Best & Worst & Med. & Mean & Std & \\',
    r'\midrule',
]

prev_type = None
for _, row in df.iterrows():
    if row['Type'] != prev_type:
        lines.append(r'\midrule')
        lines.append(r'\multicolumn{14}{l}{\textit{' + row['Type'] + r'}} \\')
        prev_type = row['Type']

    sig  = row[f'sig_{col}']
    bold = (sig == '+')
    bo, bc = (r'\textbf{', '}') if bold else ('', '')
    fr_u  = row['fr_nb1']
    fr_mn = row[f'fr_{col}']
    fr_str = f'{fr_u:.0f}→{fr_mn:.0f}'

    lines.append(
        f"{row['Name']} & {row['Type']} & {fr_str}"
        f" & {fmt_e(row['nb1_best'])} & {fmt_e(row['nb1_worst'])}"
        f" & {fmt_e(row['nb1_median'])} & {fmt_e(row['nb1_mean'])} & {fmt_e(row['nb1_std'])}"
        f" & {bo}{fmt_e(row[f'{col}_best'])}{bc}"
        f" & {bo}{fmt_e(row[f'{col}_worst'])}{bc}"
        f" & {bo}{fmt_e(row[f'{col}_median'])}{bc}"
        f" & {bo}{fmt_e(row[f'{col}_mean'])}{bc}"
        f" & {bo}{fmt_e(row[f'{col}_std'])}{bc}"
        f" & {sig_sym(sig)} \\\\"
    )

W_ = (df[f'sig_{col}']=='+').sum()
T_ = (df[f'sig_{col}']=='=').sum()
L_ = (df[f'sig_{col}']=='-').sum()
fr_u_avg  = df['fr_nb1'].mean()
fr_mn_avg = df[f'fr_{col}'].mean()

lines += [
    r'\midrule',
    r'\multicolumn{14}{l}{W/T/L (MNABC_neighb nb=' + str(best_MNABC_neighb_nb) + r' vs MNABC nb=1): '
        + f'{W_}/{T_}/{L_}' + r' / 28} \\',
    r'\multicolumn{14}{l}{Friedman Avg.~Rank --- MNABC: ' + f'{fr_u_avg:.3f}'
        + r'\ \ MNABC_neighb: ' + f'{fr_mn_avg:.3f}' + r'} \\',
    r'\bottomrule',
    r'\end{tabular}',
    r'\begin{tablenotes}\small',
    r'\item $\bullet$: MNABC_neighb sig.\ better; $\circ$: MNABC sig.\ better; $\approx$: no sig.\ diff.\ (Wilcoxon $\alpha{=}0.05$).',
    r'\item MNABC_neighb Eq.(2\textquotesingle): multi-neighborhood anchor-shift, nb\_size=' + str(best_MNABC_neighb_nb) + r'\ dimensions (annular).',
    r'\item SN$=D=' + str(D) + r'$, MaxFEs$=' + str(PARAMS['MaxFEs']) + r'$, Runs$=30$, $\alpha=' + str(PARAMS['alpha']) + r'$.',
    r'\item Rank$_\mathrm{Fr}$: MNABC rank → MNABC_neighb rank (Friedman, lower = better).',
    r'\end{tablenotes}',
    r'\end{table*}',
    '',
    r'% Full ablation table (all nb_sizes) in supplementary material.',
]

tex = '\n'.join(lines)
with open(f'results_MNABC_neighb_Param_CEC2013_D{D}/latex_table_cec2013.tex', 'w') as f:
    f.write(tex)
print('✓ latex_table_cec2013.tex généré')
print()
print(tex[:1200])
print('...')

In [ ]:
lines2 = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Friedman Ranks — MNABC_neighb nb\_size Ablation on CEC 2013 (D=30, 30 runs)}',
    r'\label{tab:ablation_friedman}',
    r'\setlength{\tabcolsep}{4pt}',
    r'\begin{tabular}{llrrrrrr}',
    r'\toprule',
    r'Func & Type & nb=1 & nb=2 & nb=3 & nb=5 & nb=7 & nb=9 \\',
    r'\midrule',
]

prev_type = None
for _, row in df.iterrows():
    if row['Type'] != prev_type:
        lines2.append(r'\midrule')
        lines2.append(r'\multicolumn{8}{l}{\textit{' + row['Type'] + r'}} \\')
        prev_type = row['Type']
    best_rank_col = min(['fr_nb1','fr_nb2','fr_nb3','fr_nb5','fr_nb7','fr_nb9'],
                        key=lambda c: row[c])
    vals = []
    for col_name in ['fr_nb1','fr_nb2','fr_nb3','fr_nb5','fr_nb7','fr_nb9']:
        v = f'{row[col_name]:.1f}'
        if col_name == best_rank_col:
            v = r'\textbf{' + v + r'}'
        vals.append(v)
    lines2.append(f"{row['Name']} & {row['Type']} & {' & '.join(vals)} \\\\")

avg_line = ['Avg.']
for nb in NB_SIZES:
    avg_val = df[f'fr_nb{nb}'].mean()
    avg_line.append(f'{avg_val:.3f}')
best_avg_col = min(NB_SIZES, key=lambda nb: df[f'fr_nb{nb}'].mean())
avg_idx = NB_SIZES.index(best_avg_col) + 1                     
avg_vals_str = []
for nb in NB_SIZES:
    v = f'{df[f"fr_nb{nb}"].mean():.3f}'
    if nb == best_avg_col:
        v = r'\textbf{' + v + r'}'
    avg_vals_str.append(v)

lines2 += [
    r'\midrule',
    f'\\textbf{{Avg.}} & — & {" & ".join(avg_vals_str)} \\\\',
    r'\bottomrule',
    r'\end{tabular}',
    r'\begin{tablenotes}\small',
    r'\item Bold = best rank per row/column. nb=1 : MNABC baseline.',
    r'\end{tablenotes}',
    r'\end{table}',
]

tex2 = '\n'.join(lines2)
with open(f'results_MNABC_neighb_Param_CEC2013_D{D}/latex_ablation_friedman.tex', 'w') as f:
    f.write(tex2)
print('✓ latex_ablation_friedman.tex généré')
print()
print(tex2[:800])
print('...')

In [ ]:
ALPHA_VALUES = [0.3, 0.5, 0.6, 0.7, 0.9]
HUFS_RATIOS  = [0.1, 0.2, 0.3]

SENS_NB_SIZE = best_MNABC_neighb_nb if 'best_MNABC_neighb_nb' in globals() else 3

SENS_FUNC_IDS = list(range(1, 29))

SENS_RUNS = 30                                                                      

SENS_PARAMS = dict(
    ALPHA_VALUES = ALPHA_VALUES,
    HUFS_RATIOS  = HUFS_RATIOS,
    NB_SIZE      = SENS_NB_SIZE,
    FUNC_IDS     = SENS_FUNC_IDS,
    RUNS         = SENS_RUNS,
    SN           = SN,
    limit        = limit_,
    MaxFEs       = MaxFEs_,
)

COMBOS = [(a, h) for a in ALPHA_VALUES for h in HUFS_RATIOS]

print('Analyse de sensibilité — MNABC_neighb (nb_size fixé) :')
print(f'  α ∈ {ALPHA_VALUES}')
print(f'  hufs_ratio ∈ {HUFS_RATIOS}')
print(f'  nb_size fixé        = {SENS_NB_SIZE}  (meilleure config. ablation)')
print(f'  Fonctions testées   = {SENS_FUNC_IDS}  ({len(SENS_FUNC_IDS)}/28)')
print(f'  Combinaisons (α,hufs_ratio) = {len(COMBOS)}')
print(f'  Runs par combinaison         = {SENS_RUNS}')
print(f'  Total runs estimé   = {len(SENS_FUNC_IDS)*len(COMBOS)*SENS_RUNS:,}')
print('  ⚠ Analyse complète (28 fonctions, 30 runs) — temps de calcul conséquent attendu.')

In [ ]:
sens_results = {}

In [ ]:
t0_sens = time.time()

for fid in SENS_FUNC_IDS:
    fname = CEC2013_INFO[fid][0]
    ftype = CEC2013_INFO[fid][2]

    if fid not in sens_results:
        sens_results[fid] = {combo: {'errors': []} for combo in COMBOS}

    done = [c for c in COMBOS if len(sens_results[fid][c]['errors']) >= SENS_RUNS]
    if len(done) == len(COMBOS):
        means = {c: np.mean(sens_results[fid][c]['errors']) for c in COMBOS}
        best_c = min(means, key=means.get)
        print(f'{fname} ({ftype:10s}) [complet] best=(α={best_c[0]}, hufs={best_c[1]}) '
              f'→ {means[best_c]:.2e}')
        continue

    t_f = time.time()
    for run in range(SENS_RUNS):
        seed = run * 211 + fid * 53                                                 

        for (a, h) in COMBOS:
            if len(sens_results[fid][(a, h)]['errors']) > run:
                continue                

            opt = MNABC_neighb(
                fid, SN=SENS_PARAMS['SN'], MaxFEs=SENS_PARAMS['MaxFEs'],
                limit=SENS_PARAMS['limit'], D=D,
                alpha=a, hufs_ratio=h, nb_size=SENS_NB_SIZE)

            err, _ = opt.run(seed=seed)
            sens_results[fid][(a, h)]['errors'].append(float(err))

        if (run+1) % 5 == 0 or (run+1) == SENS_RUNS:
            print(f'  {fname} ({ftype:10s}) run {run+1:2d}/{SENS_RUNS}', flush=True)

    elapsed = time.time() - t_f
    means = {c: np.mean(sens_results[fid][c]['errors']) for c in COMBOS}
    best_c = min(means, key=means.get)
    print(f'{fname} TERMINÉ [{elapsed:.0f}s]  best=(α={best_c[0]}, hufs={best_c[1]}) '
          f'→ {means[best_c]:.2e}')

print(f'\n✓ Analyse de sensibilité terminée en {(time.time()-t0_sens)/3600:.2f}h')

In [ ]:
sens_rows = []
for fid in SENS_FUNC_IDS:
    if fid not in sens_results: continue
    fname = CEC2013_INFO[fid][0]
    ftype = CEC2013_INFO[fid][2]

    means_combo = [np.mean(sens_results[fid][c]['errors']) for c in COMBOS]
    ranks_combo = list(rankdata(means_combo))                

    row = {'FuncID': fid, 'Name': fname, 'Type': ftype}
    for (combo, m, r) in zip(COMBOS, means_combo, ranks_combo):
        a, h = combo
        row[f'mean_a{a}_h{h}'] = m
        row[f'rank_a{a}_h{h}'] = r
    best_combo = COMBOS[int(np.argmin(means_combo))]
    row['best_alpha'], row['best_hufs'] = best_combo
    sens_rows.append(row)

df_sens = pd.DataFrame(sens_rows)

avg_rank_grid = np.zeros((len(ALPHA_VALUES), len(HUFS_RATIOS)))
for i, a in enumerate(ALPHA_VALUES):
    for j, h in enumerate(HUFS_RATIOS):
        avg_rank_grid[i, j] = df_sens[f'rank_a{a}_h{h}'].mean()

best_i, best_j = np.unravel_index(np.argmin(avg_rank_grid), avg_rank_grid.shape)
best_alpha_global = ALPHA_VALUES[best_i]
best_hufs_global  = HUFS_RATIOS[best_j]

print('═'*70)
print(f'RANG MOYEN DE FRIEDMAN — grille α × hufs_ratio ({len(SENS_FUNC_IDS)} fonctions)')
print('═'*70)
header = '  α \\\\ hufs_ratio  | ' + ' | '.join(f'{h:>6.1f}' for h in HUFS_RATIOS)
print(header)
print('-'*len(header))
for i, a in enumerate(ALPHA_VALUES):
    line = f'  α = {a:<10.1f} | ' + ' | '.join(f'{avg_rank_grid[i,j]:6.2f}' for j in range(len(HUFS_RATIOS)))
    print(line)

print()
print(f'✓ Meilleure combinaison globale : α={best_alpha_global}, hufs_ratio={best_hufs_global} '
      f'(rang moyen={avg_rank_grid[best_i,best_j]:.3f})')

try:
    samples = [[df_sens.loc[df_sens['FuncID']==fid, f'mean_a{a}_h{h}'].values[0]
                for fid in SENS_FUNC_IDS] for (a, h) in COMBOS]
    stat, p_friedman = friedmanchisquare(*samples)
    print(f'\nFriedman χ² = {stat:.3f}, p-value = {p_friedman:.4e} '
          f'({"significatif" if p_friedman < 0.05 else "non significatif"} à α=0.05)')
except Exception as e:
    print(f'\nFriedman test non calculable : {e}')

print('\nMeilleure combinaison par type de fonction :')
for ftype in df_sens['Type'].unique():
    sub = df_sens[df_sens['Type'] == ftype]
    rank_cols = [f'rank_a{a}_h{h}' for (a, h) in COMBOS]
    avg_by_combo = sub[rank_cols].mean()
    best_col = avg_by_combo.idxmin()
    print(f'  {ftype:10s} : meilleur = {best_col}  (rang moyen={avg_by_combo.min():.3f})')

df_sens.to_csv(f'{SENS_DIR}/sensitivity_full_stats.csv', index=False)
print(f'\n✓ Sauvegardé → {SENS_DIR}/sensitivity_full_stats.csv')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
im = ax.imshow(avg_rank_grid, cmap='RdYlGn_r', aspect='auto',
               vmin=1, vmax=len(COMBOS))
ax.set_xticks(range(len(HUFS_RATIOS))); ax.set_xticklabels([f'{h:.1f}' for h in HUFS_RATIOS])
ax.set_yticks(range(len(ALPHA_VALUES))); ax.set_yticklabels([f'{a:.1f}' for a in ALPHA_VALUES])
ax.set_xlabel('hufs_ratio'); ax.set_ylabel('α')
ax.set_title(f'Rang moyen de Friedman (n={len(SENS_FUNC_IDS)} fonctions)\nnb_size={SENS_NB_SIZE} fixe')
for i in range(len(ALPHA_VALUES)):
    for j in range(len(HUFS_RATIOS)):
        ax.text(j, i, f'{avg_rank_grid[i,j]:.2f}', ha='center', va='center',
                 fontsize=10, color='black')
ax.add_patch(plt.Rectangle((best_j-0.5, best_i-0.5), 1, 1, fill=False,
                            edgecolor='black', lw=2.5))
plt.colorbar(im, ax=ax, label='Rang moyen (1=meilleur)')

ax2 = axes[1]
log_err_grid = np.zeros((len(ALPHA_VALUES), len(HUFS_RATIOS)))
for i, a in enumerate(ALPHA_VALUES):
    for j, h in enumerate(HUFS_RATIOS):
        vals = df_sens[f'mean_a{a}_h{h}']
        log_err_grid[i, j] = np.mean(np.log10(np.maximum(vals, 1e-12)))

im2 = ax2.imshow(log_err_grid, cmap='RdYlGn_r', aspect='auto')
ax2.set_xticks(range(len(HUFS_RATIOS))); ax2.set_xticklabels([f'{h:.1f}' for h in HUFS_RATIOS])
ax2.set_yticks(range(len(ALPHA_VALUES))); ax2.set_yticklabels([f'{a:.1f}' for a in ALPHA_VALUES])
ax2.set_xlabel('hufs_ratio'); ax2.set_ylabel('α')
ax2.set_title('Moyenne de log₁₀(erreur) sur les fonctions testées')
for i in range(len(ALPHA_VALUES)):
    for j in range(len(HUFS_RATIOS)):
        ax2.text(j, i, f'{log_err_grid[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im2, ax=ax2, label='log₁₀(Mean Error) moyen')

plt.tight_layout()
plt.savefig(f'{SENS_DIR}/fig_sensitivity_heatmaps.pdf', bbox_inches='tight')
plt.show()
print(f'✓ {SENS_DIR}/fig_sensitivity_heatmaps.pdf')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

HR_COLORS = {0.1: '#2980B9', 0.2: '#0F6E56', 0.3: '#C0392B'}
A_COLORS  = plt.cm.viridis(np.linspace(0.15, 0.9, len(ALPHA_VALUES)))

ax = axes[0]
for j, h in enumerate(HUFS_RATIOS):
    ys = [avg_rank_grid[i, j] for i in range(len(ALPHA_VALUES))]
    ax.plot(ALPHA_VALUES, ys, marker='o', lw=2, color=HR_COLORS.get(h, None),
             label=f'hufs_ratio={h}')
ax.set_xlabel('α (balance utilité fitness/diversité)')
ax.set_ylabel('Rang moyen de Friedman (1=meilleur)')
ax.set_title("Sensibilité à α — par hufs_ratio")
ax.legend(); ax.invert_yaxis()

ax2 = axes[1]
for i, a in enumerate(ALPHA_VALUES):
    ys = [avg_rank_grid[i, j] for j in range(len(HUFS_RATIOS))]
    ax2.plot(HUFS_RATIOS, ys, marker='s', lw=2, color=A_COLORS[i], label=f'α={a}')
ax2.set_xlabel('hufs_ratio')
ax2.set_ylabel('Rang moyen de Friedman (1=meilleur)')
ax2.set_title('Sensibilité à hufs_ratio — par α')
ax2.legend(); ax2.invert_yaxis()

plt.tight_layout()
plt.savefig(f'{SENS_DIR}/fig_sensitivity_curves.pdf', bbox_inches='tight')
plt.show()
print(f'✓ {SENS_DIR}/fig_sensitivity_curves.pdf')

In [ ]:
lines_sens = [
    r'\begin{table}[htbp]',
    r'\centering',
    r'\caption{Sensitivity analysis of $\alpha$ and hufs\_ratio (MNABC_neighb, nb\_size=' + str(SENS_NB_SIZE) + r')}',
    r'\label{tab:sensitivity_alpha_hufs}',
    r'\begin{tabular}{l' + 'r'*len(HUFS_RATIOS) + '}',
    r'\toprule',
    r'$\alpha \backslash$ hufs\_ratio & ' + ' & '.join(f'{h:.1f}' for h in HUFS_RATIOS) + r' \\',
    r'\midrule',
]
for i, a in enumerate(ALPHA_VALUES):
    vals = []
    for j, h in enumerate(HUFS_RATIOS):
        v = f'{avg_rank_grid[i, j]:.2f}'
        if (i, j) == (best_i, best_j):
            v = r'\textbf{' + v + r'}'
        vals.append(v)
    lines_sens.append(f'{a:.1f} & ' + ' & '.join(vals) + r' \\')
lines_sens += [
    r'\bottomrule',
    r'\end{tabular}',
    r'\begin{tablenotes}\small',
    r'\item Values = average Friedman rank (lower is better) over '
    + str(len(SENS_FUNC_IDS)) + r' CEC 2013 functions, '
    + str(SENS_RUNS) + r' runs. Bold = best combination.',
    r'\end{tablenotes}',
    r'\end{table}'
]
tex_sens = '\n'.join(lines_sens)
with open(f'{SENS_DIR}/table_sensitivity.tex', 'w') as f:
    f.write(tex_sens)
print(tex_sens)
print(f'\n✓ {SENS_DIR}/table_sensitivity.tex')

In [ ]:
import zipfile

print('═'*70)
print('RÉSUMÉ FINAL — MNABC_neighb vs MNABC · CEC 2013')
print('═'*70)
print(f'  D = {D},  SN = {SN},  MaxFEs = {MaxFEs_:,},  Runs = {RUNS}')
print()
print('Friedman Average Rank par configuration (ablation nb_size) :')
for nb in NB_SIZES:
    label = 'MNABC (baseline)   ' if nb==1 else f'MNABC_neighb nb_size={nb}  '
    print(f'  nb={nb}  {label}: {df[f"fr_nb{nb}"].mean():.4f}')
print()
print(f'Meilleure config globale (nb_size) : nb_size={best_MNABC_neighb_nb}')
print()
print('W/T/L (chaque MNABC_neighb config vs MNABC baseline, 28 fonctions) :')
for nb in [2,3,5,7,9]:
    W = (df[f'sig_nb{nb}']=='+').sum()
    T = (df[f'sig_nb{nb}']=='=').sum()
    L = (df[f'sig_nb{nb}']=='-').sum()
    print(f'  nb={nb}: W={W:2d} T={T:2d} L={L:2d}')

if 'avg_rank_grid' in globals():
    print()
    print('─'*70)
    print(f'Analyse de sensibilité (MNABC_neighb, nb_size={SENS_NB_SIZE} fixe, '
          f'{len(SENS_FUNC_IDS)} fonctions, {SENS_RUNS} runs) :')
    print(f'  α testés          : {ALPHA_VALUES}')
    print(f'  hufs_ratio testés : {HUFS_RATIOS}')
    print(f'  → Meilleure combinaison : α={best_alpha_global}, hufs_ratio={best_hufs_global} '
          f'(rang moyen={avg_rank_grid[best_i,best_j]:.3f})')

zip_path = 'MNABC_neighb_vs_MNABC_CEC2013_results.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(glob.glob(f'results_MNABC_neighb_Param_CEC2013_D{D}/**/*.*', recursive=True)):
        if os.path.isfile(f):
            zf.write(f)

print(f'\n✓ Archive → {zip_path}')
print('Contenu :')
for f in sorted(glob.glob(f'results_MNABC_neighb_Param_CEC2013_D{D}/**/*.*', recursive=True)):
    if os.path.isfile(f):
        sz = os.path.getsize(f)/1024
        print(f'  {f:75s}  ({sz:.1f} Ko)')

try:
    from google.colab import files
    files.download(zip_path)
    print('\n↓ Téléchargement Colab lancé.')
except ImportError:
    print(f'\n(Local : archive disponible dans {os.path.abspath(zip_path)})')